In [6]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [7]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
682,Um .... a serious film about troubled teens in...,positive
193,This movie is bad. I saw the rated and the unr...,negative
744,Everything about this film was terrible. To st...,negative
672,USA The Movie is like this: You take a nap on ...,positive
474,I really don't understand who this movie is ai...,negative


In [8]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:32: SyntaxWarning: invalid escape sequence '\s'
<>:32: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Aditya Ghige\AppData\Local\Temp\ipykernel_3276\3798096103.py:32: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [10]:
df = normalize_text(df)
df.head()

,review,sentiment
682,um serious film troubled teen singapore countr...,positive
193,movie bad saw rated unrated version terrible n...,negative
744,everything film terrible start film pretty goo...,negative
672,usa movie like this take nap long hot sunday a...,positive
474,really understand movie aimed at absurdity it ...,negative


In [9]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package stopwords to C:\Users\Aditya
[nltk_data]     Ghige\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to C:\Users\Aditya
[nltk_data]     Ghige\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to C:\Users\Aditya
[nltk_data]     Ghige\AppData\Roaming\nltk_data...


True

In [11]:
df['sentiment'].value_counts()

sentiment
negative    260
positive    240
Name: count, dtype: int64

In [12]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [13]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
682,um serious film troubled teen singapore countr...,1
193,movie bad saw rated unrated version terrible n...,0
744,everything film terrible start film pretty goo...,0
672,usa movie like this take nap long hot sunday a...,1
474,really understand movie aimed at absurdity it ...,0


In [14]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [15]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/Ghige/mlops_capstone_project.mlflow')
dagshub.init(repo_owner='ADITYA GHIGE', repo_name='mlops_capstone_project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\yt ml ops\mlops_capstone_project\adi\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=7e01c7a3-147c-43d0-a829-5c35022b4ce7&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=7a728882b186dda2721bd3b19d417c0bf4bbf0efc51cbbf60b0626d277a8de0e




Accessing as Ghige

Repository YT-Capstone-Project doesn't exist, creating it under current user.

Initialized MLflow to track repo "ADITYA GHIGE/YT-Capstone-Project"

Repository ADITYA GHIGE/YT-Capstone-Project initialized!

2026/01/25 04:52:10 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/d03ab8974a2a4f0097b1f6974301ea16', creation_time=1769296929487, experiment_id='0', last_update_time=1769296929487, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}>

In [18]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-01-25 04:58:25,841 - INFO - Starting MLflow run...
2026-01-25 04:58:26,991 - INFO - Logging preprocessing parameters...
2026-01-25 04:58:27,919 - INFO - Initializing Logistic Regression model...
2026-01-25 04:58:27,920 - INFO - Fitting the model...
2026-01-25 04:58:27,936 - INFO - Model training complete.
2026-01-25 04:58:27,937 - INFO - Logging model parameters...
2026-01-25 04:58:28,260 - INFO - Making predictions...
2026-01-25 04:58:28,261 - INFO - Calculating evaluation metrics...
2026-01-25 04:58:28,267 - INFO - Logging evaluation metrics...
2026-01-25 04:58:29,515 - INFO - Saving and logging the model...
2026/01/25 04:58:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-01-25 04:58:43,265 - INFO - Model training and logging completed in 16.27 seconds.
2026-01-25 04:58:43,265 - INFO - Accuracy: 0.672
2026-01-25 04:58:43,266 - INFO - Precision: 0.6333333333333333
2026-01-25 04:58:43,266 - INFO - Recall: 0.6666666666666666
2026-01-25

🏃 View run indecisive-cow-663 at: https://dagshub.com/Ghige/mlops_capstone_project.mlflow/#/experiments/0/runs/3f3a342a4ee943d9abd44dd3ea29556d
🧪 View experiment at: https://dagshub.com/Ghige/mlops_capstone_project.mlflow/#/experiments/0
